<a href="https://colab.research.google.com/github/342olive/scraped-data/blob/main/NLP-Applications/09_quick_run_unindented_pipeline_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# config.py
import transformers
# this is the maximum number of tokens in the sentence
MAX_LEN = 512
# batch sizes is small because model is huge!
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 4
# let's train for a maximum of 10 epochs
EPOCHS = 10
# define path to BERT model files
BERT_PATH = "../input/bert_base_uncased/"
# this is where you want to save the model
MODEL_PATH = "model.bin"
# training file
TRAINING_FILE = "../input/imdb.csv"
# define the tokenizer
# we use tokenizer and model
# from huggingface's transformers
TOKENIZER = transformers.BertTokenizer.from_pretrained(
BERT_PATH,
do_lower_case=True
)

In [ ]:
# dataset.py
import config
import torch


class BERTDataset:
    def __init__(self, review, target):
        """
        :param review: list or numpy array of strings
        :param targets: list or numpy array which is binary
        """
        self.review = review
        self.target = target

        # we fetch max len and tokenizer from config.py
        self.tokenizer = config.TOKENIZER
        self.max_len = config.MAX_LEN

    def __len__(self):
        # this returns the length of dataset
        return len(self.review)

    def __getitem__(self, item):
        # for a given item index, return a dictionary of inputs
        review = str(self.review[item])
        review = " ".join(review.split())

        # encode the review
        inputs = self.tokenizer.encode_plus(
            review,
            None,
            add_special_tokens=True,


In [ ]:
# engine.py
import torch
import torch.nn as nn


def loss_fn(outputs, targets):
    """
    This function returns the loss.
    :param outputs: output from the model (real numbers)
    :param targets: input targets (binary)
    """
    return nn.BCEWithLogitsLoss()(outputs, targets.view(-1, 1))


def train_fn(data_loader, model, optimizer, device, scheduler):
    """
    This is the training function which trains for one epoch
    :param data_loader: it is the torch dataloader object
    :param model: torch model, bert in our case
    :param optimizer: adam, sgd, etc
    :param device: can be cpu or cuda
    :param scheduler: learning rate scheduler
    """

    # put the model in training mode
    model.train()

    # loop over all batches
    for d in data_loader:
        # extract ids, token type ids and mask
        ids = d["ids"]
        token_type_ids = d["token_type_ids"]
        mask = d["mask"]
        targets = d["targets"]

        # move everything to specified device
        ids = ids.to(device, dtype=torch.long)
        token_type_ids = token_type_ids.to(device, dtype=torch.long)
        mask = mask.to(device, dtype=torch.long)
        targets = targets.to(device, dtype=torch.float)

        # zero-grad the optimizer
        optimizer.zero_grad()

        # pass through the model
        outputs = model(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )

        # calculate loss
        loss = loss_fn(outputs, targets)

        # backward step
        loss.backward()

        # optimizer step
        optimizer.step()

        # scheduler step
        scheduler.step()


def eval_fn(data_loader, model, device):
    """
    this is the validation function that generates
    predictions on validation data
    :param data_loader: it is the torch dataloader object
    :param model: torch model, bert in our case
    :param device: can be cpu or cuda
    :return: output and targets
    """

    # put model in eval mode
    model.eval()

    fin_targets = []
    fin_outputs = []

    # no grad context
    with torch.no_grad():
        for d in data_loader:
            ids = d["ids"]
            token_type_ids = d["token_type_ids"]
            mask = d["mask"]
            targets = d["targets"]

            ids = ids.to(device, dtype=torch.long)
            token_type_ids = token_type_ids.to(device, dtype=torch.long)
            mask = mask.to(device, dtype=torch.long)
            targets = targets.to(device, dtype=torch.float)

            outputs = model(
                ids=ids,
                mask=mask,
                token_type_ids=token_type_ids
            )

            # move to cpu
            targets = targets.cpu().detach()
            fin_targets.extend(targets.numpy().tolist())

            outputs = torch.sigmoid(outputs).cpu().detach()
            fin_outputs.extend(outputs.numpy().tolist())

    return fin_outputs, fin_targets


In [ ]:
# model.py
import config
import transformers
import torch.nn as nn


class BERTBaseUncased(nn.Module):
    def __init__(self):
        super(BERTBaseUncased, self).__init__()

        # we fetch the model from the BERT_PATH defined in
        # config.py
        self.bert = transformers.BertModel.from_pretrained(
            config.BERT_PATH
        )

        # add a dropout for regularization
        self.bert_drop = nn.Dropout(0.3)

        # a simple linear layer for output
        # yes, there is only one output
        self.out = nn.Linear(768, 1)

    def forward(self, ids, mask, token_type_ids):
        # BERT in its default settings returns two outputs
        # last hidden state and output of bert pooler layer
        # we use the output of the pooler which is of the size
        # (batch_size, hidden_size)

        _, o2 = self.bert(
            ids,
            attention_mask=mask,
            token_type_ids=token_type_ids
        )

        # pass through dropout layer
        bo = self.bert_drop(o2)

        # pass through linear layer
        output = self.out(bo)

        # return output
        return output


In [ ]:
# train.py
import config
import dataset
import engine
import torch
import pandas as pd
import torch.nn as nn
import numpy as np

from model import BERTBaseUncased
from sklearn import model_selection
from sklearn import metrics
from transformers import AdamW
from transformers import get_linear_schedule_with_warmup


def train():
    # this function trains the model

    # read the training file and fill NaN values with "none"
    dfx = pd.read_csv(config.TRAINING_FILE).fillna("none")

    # sentiment = 1 if its positive else 0
    dfx.sentiment = dfx.sentiment.apply(
        lambda x: 1 if x == "positive" else 0
    )

    # split the data into training and validation
    df_train, df_valid = model_selection.train_test_split(
        dfx,
        test_size=0.1,
        random_state=42,
        stratify=dfx.sentiment.values
    )

    # reset index
    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)

    # training dataset
    train_dataset = dataset.BERTDataset(
        review=df_train.review.values,
        target=df_train.sentiment.values
    )

    # training dataloader
    train_data_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config.TRAIN_BATCH_SIZE,
        num_workers=4
    )

    # validation dataset
    valid_dataset = dataset.BERTDataset(
        review=df_valid.review.values,
        target=df_valid.sentiment.values
    )

    # validation dataloader
    valid_data_loader = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=config.VALID_BATCH_SIZE,
        num_workers=1
    )

    # device
    device = torch.device("cuda")

    # model
    model = BERTBaseUncased()
    model.to(device)

    # optimizer parameters
    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

    optimizer_parameters = [
        {
            "params": [
                p for n, p in param_optimizer
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.001,
        },
        {
            "params": [
                p for n, p in param_optimizer
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    # number of training steps
    num_train_steps = int(
        len(df_train) / config.TRAIN_BATCH_SIZE * config.EPOCHS
    )

    # optimizer
    optimizer = AdamW(optimizer_parameters, lr=3e-5)

    # scheduler
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=num_train_steps
    )

    # DataParallel
    model = nn.DataParallel(model)

    # training loop
    best_accuracy = 0

    for epoch in range(config.EPOCHS):
        engine.train_fn(
            train_data_loader,
            model,
            optimizer,
            device,
            scheduler
        )

        outputs, targets = engine.eval_fn(
            valid_data_loader,
            model,
            device
        )

        outputs = np.array(outputs) >= 0.5
        accuracy = metrics.accuracy_score(targets, outputs)

        print(f"Accuracy Score = {accuracy}")

        if accuracy > best_accuracy:
            torch.save(model.state_dict(), config.MODEL_PATH)
            best_accuracy = accuracy


if __name__ == "__main__":
    train()


Code snippets that I tweaked during learning

Dataset: Use padding='max_length' and truncation=True (as we discussed).

Model: Use output.pooler_output instead of output[1].

Training: Use torch.optim.AdamW instead of the one from the transformers library.

In [ ]:
  # model.py   # OLD
  # encode the review
        inputs = self.tokenizer.encode_plus(
            review,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True
        )



In [ ]:
 # MODERN
inputs = self.tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    max_length=self.max_len,
    padding='max_length',     # REPLACES pad_to_max_length=True
    truncation=True,          # MUST be True if using max_length
    return_attention_mask=True,
    return_tensors='pt'
)

In [ ]:
# Inside the train() function
for bi, d in enumerate(data_loader):
    # 'd' contains the IDs and Masks from your Dataset's __getitem__

    # THIS is where the forward() function in model.py is called:
    outputs = model(
        ids=d["ids"],
        mask=d["mask"],
        token_type_ids=d["token_type_ids"]
    )

    # This is where the math happens to check how "wrong" the model was
    loss = loss_fn(outputs, d["targets"])

In [ ]:
# model.py
def forward(self, ids, mask, token_type_ids):
        # 1. Call the modern model
        outputs = self.bert(
            input_ids=ids,
            attention_mask=mask,
            token_type_ids=token_type_ids
        )

        # 2. ALIAS: Map the new object attribute to the author's 'o2'
        o2 = outputs.pooler_output

        # 3. The rest of the logic remains identical to the author's names
        bo = self.bert_drop(o2)
        output = self.out(bo)

        return output

In [ ]:
# Modern way (v4.35.0 compatible)
outputs = self.bert(ids, mask, token_type_ids)
o2 = outputs.pooler_output  # This extracts the [CLS] tensor you need

In [ ]:
# This saves the content of this cell to 'model_utils.py'
import os

code = """
def compare_models(bert_results, lstm_results):
    # your comparison logic here
    pass
"""

with open("model_utils.py", "w") as f:
    f.write(code)

In [ ]:
%%writefile engine_v2.py
import torch
import torch.nn as nn
from tqdm import tqdm

def loss_fn(outputs, targets):
    """
    Computes Binary Cross Entropy with Logits.
    targets.view(-1, 1) ensures the shape matches BERT's [batch, 1] output.
    """
    return nn.BCEWithLogitsLoss()(outputs, targets.view(-1, 1))

def train_fn(data_loader, model_v2, optimizer, device, scheduler):
    model_v2.train()

    # tk0 is our progress bar. 'total' helps it calculate the ETA.
    tk0 = tqdm(data_loader, total=len(data_loader), desc="Training")

    for d in tk0:
        ids = d["ids"].to(device, dtype=torch.long)
        token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
        mask = d["mask"].to(device, dtype=torch.long)
        targets = d["targets"].to(device, dtype=torch.float)

        # Faster than .zero_grad()
        optimizer.zero_grad(set_to_none=True)

        # Named arguments match our modernized model.py forward()
        outputs = model(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )

        loss = loss_fn(outputs, targets)
        loss.backward()

        optimizer.step()
        scheduler.step()

        # Shows live loss next to the progress bar
        tk0.set_postfix(loss=loss.item())

def eval_fn(data_loader, model_v2, device):
    model_v2.eval()

    fin_targets = []
    fin_outputs = []

    # Progress bar for validation too!
    tk0 = tqdm(data_loader, total=len(data_loader), desc="Evaluating")

    with torch.no_grad():
        for d in tk0:
            ids = d["ids"].to(device, dtype=torch.long)
            token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
            mask = d["mask"].to(device, dtype=torch.long)
            targets = d["targets"].to(device, dtype=torch.float)

            outputs = model_v2(
                ids=ids,
                mask=mask,
                token_type_ids=token_type_ids
            )

            # Accumulate targets and sigmoid-transformed probabilities
            targets = targets.cpu().detach()
            fin_targets.extend(targets.numpy().tolist())

            # Convert raw BERT logits to 0-1 probabilities for later metrics
            outputs = torch.sigmoid(outputs).cpu().detach()
            fin_outputs.extend(outputs.numpy().tolist())

    return fin_outputs, fin_targets

In [ ]:
# writeup for shape mismatches
# "I took an older implementation and refactored it to work with Transformers v4.35, resolving several shape mismatches and API deprecations."

In [ ]:


#This is to debug the model to see what it recieves
# model_v2.py
def forward(self, ids, mask, token_type_ids):
    # ADD THESE TEMPORARY LINES:
    print(f"DEBUG: ids type: {type(ids)}")
    if ids is not None:
        print(f"DEBUG: ids shape: {ids.shape}")
    else:
        print("DEBUG: ids is NONE!")

    outputs = self.bert(
        input_ids=ids,
        attention_mask=mask,
        token_type_ids=token_type_ids
    )
    # ... rest of code

The out of sync bert pipeline


In [ ]:
%%writefile engine_v2.py
import torch
import torch.nn as nn
from tqdm import tqdm

def loss_fn(outputs, targets):
    """
    Computes Binary Cross Entropy with Logits.
    targets.view(-1, 1) ensures the shape matches BERT's [batch, 1] output.
    """
    return nn.BCEWithLogitsLoss()(outputs, targets.view(-1, 1))

def train_fn(data_loader, model_v2, optimizer, device, scheduler):
    model_v2.train()

    # tk0 is our progress bar. 'total' helps it calculate the ETA.
    tk0 = tqdm(data_loader, total=len(data_loader), desc="Training")

    for d in tk0:
        ids = d["ids"].to(device, dtype=torch.long)
        token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
        mask = d["mask"].to(device, dtype=torch.long)
        targets = d["targets"].to(device, dtype=torch.float)

        # Faster than .zero_grad()
        optimizer.zero_grad(set_to_none=True)

        # Named arguments match our modernized model.py forward()
        outputs = model_v2(
            ids=ids,
            mask=mask,
            token_type_ids=token_type_ids
        )

        loss = loss_fn(outputs, targets)
        loss.backward()

        optimizer.step()
        scheduler.step()

        # Shows live loss next to the progress bar
        tk0.set_postfix(loss=loss.item())

def eval_fn(data_loader, model_v2, device):
    model_v2.eval() # <--- Must match

    fin_targets = []
    fin_outputs = []

    tk0 = tqdm(data_loader, total=len(data_loader), desc="Evaluating")

    with torch.no_grad():
        for d in tk0:
            ids = d["ids"].to(device, dtype=torch.long)
            token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
            mask = d["mask"].to(device, dtype=torch.long)
            targets = d["targets"].to(device, dtype=torch.float)

            # THIS is the critical line
            outputs = model_v2(
                ids=ids,
                mask=mask,
                token_type_ids=token_type_ids
            )

            targets = targets.cpu().detach()
            fin_targets.extend(targets.numpy().tolist())

            outputs = torch.sigmoid(outputs).cpu().detach()
            fin_outputs.extend(outputs.numpy().tolist())

    return fin_outputs, fin_targets

In [ ]:
%%writefile dataset_v2.py
# dataset.py
import config
import torch


class BERTDataset:
    def __init__(self, review, target):
        """
        :param review: list or numpy array of strings
        :param targets: list or numpy array which is binary
        """
        self.review = review
        self.target = target

        # we fetch max len and tokenizer from config.py
        self.tokenizer = config.TOKENIZER
        self.max_len = config.MAX_LEN

    def __len__(self):
        # this returns the length of dataset
        return len(self.review)

    def __getitem__(self, item):
        # for a given item index, return a dictionary of inputs
        review = str(self.review[item])
        review = " ".join(review.split())

        #ADDED
        inputs = self.tokenizer.encode_plus(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',     # REPLACES pad_to_max_length=True
            truncation=True,          # MUST be True if using max_length
            return_attention_mask=True,
            return_tensors='pt'
        )

        # ids are ids of tokens generated after tokenizing reviews
        ids = inputs["input_ids"]

        # mask is 1 where we have input and 0 where we have padding
        mask = inputs["attention_mask"]

        # token type ids behave the same way as mask in this case
        token_type_ids = inputs["token_type_ids"]

        # return everything
        return {
            "ids": ids.detach().clone().to(dtype=torch.long),
            "mask": mask.detach().clone().to(dtype=torch.long),
            "token_type_ids": token_type_ids.detach().clone().to(dtype=torch.long),
            "targets": torch.tensor(self.target[item], dtype=torch.float)
        }

In [ ]:
%%writefile model_v2.py
# model.py
import config
import transformers
import torch.nn as nn


class BERTBaseUncased(nn.Module):
    def __init__(self):
        super(BERTBaseUncased, self).__init__()
        # This is where you add the download line:
        self.bert = transformers.BertModel.from_pretrained("bert-base-uncased")

        # add a dropout for regularization
        self.bert_drop = nn.Dropout(0.3)

        # a simple linear layer for output
        # yes, there is only one output
        self.out = nn.Linear(768, 1)

    def forward(self,ids, mask, token_type_ids):
        # 1. Call the modern model
        outputs = self.bert(
            input_ids=ids,
            attention_mask=mask,
            token_type_ids=token_type_ids
        ) #debugged, there is no comma after the bracket

        # 2. ALIAS: Map the new object attribute to the author's 'o2'
        o2 = outputs.pooler_output

        # 3. The rest of the logic remains identical to the author's names
        bo = self.bert_drop(o2)
        output = self.out(bo)

        return output

In [ ]:
%%writefile config_v2.py
# config.py
import transformers
# this is the maximum number of tokens in the sentence
MAX_LEN = 256
# batch sizes is small because model is huge!
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 4
# let's train for a maximum of 10 epochs
EPOCHS = 4
# define path to BERT model files
BERT_PATH = "bert-base-uncased"
# this is where you want to save the model
MODEL_PATH = "model.bin"
# training file
TRAINING_FILE = "/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"
# define the tokenizer
# we use tokenizer and model
# from huggingface's transformers
TOKENIZER = transformers.BertTokenizer.from_pretrained(
BERT_PATH,
do_lower_case=True
)

In [ ]:
%%writefile trainer_v2.py
import config_v2 as config # This maps config_v2 to the name 'config' so you don't have to rename everything else
import dataset_v2 as dataset
import engine_v2 as engine
import torch
import pandas as pd
import torch.nn as nn
import numpy as np
import os

from model_v2 import BERTBaseUncased # ADDED model_v2 changed name bacause we cannot use alias here
from sklearn import model_selection
from sklearn import metrics

# Optimization - Using the PyTorch implementation as required in 2026
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

def trainer_v2():
    # this function trains the model
    dfx = pd.read_csv(config.TRAINING_FILE).fillna("none")

    # sentiment = 1 if its positive else 0
    dfx.sentiment = dfx.sentiment.apply(
        lambda x: 1 if x == "positive" else 0
    )

    # split the data into training and validation
    df_train, df_valid = model_selection.train_test_split(
        dfx,
        test_size=0.1,
        random_state=42,
        stratify=dfx.sentiment.values
    )

    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)

    train_dataset = dataset.BERTDataset(
        review=df_train.review.values,
        target=df_train.sentiment.values
    )

    train_data_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config.TRAIN_BATCH_SIZE,
        num_workers=4
    )

    valid_dataset = dataset.BERTDataset(
        review=df_valid.review.values,
        target=df_valid.sentiment.values
    )

    valid_data_loader = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=config.VALID_BATCH_SIZE,
        num_workers=1
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = BERTBaseUncased()
    model.to(device)

    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

    optimizer_parameters = [
        {
            "params": [
                p for n, p in param_optimizer
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.001,
        },
        {
            "params": [
                p for n, p in param_optimizer
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    num_train_steps = int(
        len(df_train) / config.TRAIN_BATCH_SIZE * config.EPOCHS
    )

   # Modern PyTorch version
    optimizer = torch.optim.AdamW(optimizer_parameters, lr=3e-5)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=num_train_steps
    )

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    # --- Training Loop Logic ---
    best_accuracy = 0.0
    start_epoch = 0
    checkpoint_path = "/kaggle/working/checkpoint.pt"

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        best_accuracy = checkpoint["best_accuracy"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resuming from epoch {start_epoch}")

    for epoch in range(start_epoch, config.EPOCHS):
        engine.train_fn(
            train_data_loader,
            model,
            optimizer,
            device,
            scheduler
        )

        outputs, targets = engine.eval_fn(
            valid_data_loader,
            model,
            device
        )

        outputs = np.array(outputs) >= 0.5
        accuracy = metrics.accuracy_score(targets, outputs)

        print(f"Epoch {epoch} - Accuracy Score = {accuracy}")

        if accuracy > best_accuracy:
            torch.save(model.state_dict(), config.MODEL_PATH)
            best_accuracy = accuracy

        # Save checkpoint every epoch
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_accuracy": best_accuracy
        }, checkpoint_path)

if __name__ == "__main__":
    trainer_v2()

In [ ]:
import trainer_v2
trainer_v2.trainer_v2()

In [ ]:
# Replace 'my_variable' with the name of your data or model
print(locals().get('my_variable', 'Variable not found!'))